In [0]:
%sql
use catalog pl_workforce_analysis;
create schema if not exists gold;
use schema gold;


In [0]:
from pyspark.sql.functions import col, sum as _sum, year, round

fact_payroll = spark.table("pl_workforce_analysis.gold_fact.fact_payroll")
fact_finance = spark.table("pl_workforce_analysis.gold_fact.fact_finance")
dim_account = spark.table("pl_workforce_analysis.gold_dim.dim_account")

payroll_df = fact_payroll.withColumn(
    "year", year(col("pay_date"))
)

revenue_df = fact_finance.alias("ff") .join(dim_account.alias("da"), col("ff.account_id") == col("da.account_id")) .filter(col("da.account_type") == "Revenue") .withColumn("year", year(col("ff.fiscal_year")))

payroll_agg = payroll_df.groupBy("company_id", "department_id", "year") .agg(_sum("total_payroll_cost").alias("payroll"))

revenue_agg = revenue_df.groupBy("company_id", "year") .agg(_sum("final_amount").alias("revenue"))

base_df = payroll_agg.join(
    revenue_agg,
    ["company_id", "year"],
    "left"
)

cube_df = base_df.cube("company_id", "department_id", "year") .agg(
        _sum("payroll").alias("total_payroll"),
        _sum("revenue").alias("total_revenue")
    )

cube_kpi_df = cube_df.withColumn(
    "payroll_pct",
    round((col("total_payroll") / col("total_revenue")) * 100, 2)
)

final_df = cube_kpi_df.fillna({
    "company_id": "ALL",
    "department_id": "ALL",
    "year": 0
})

display(final_df)
final_df.write.mode("overwrite").saveAsTable("pl_workforce_analysis.gold.curated_dataset")

In [0]:
from pyspark.sql.functions import col, sum as _sum, avg, count, round, lag
from pyspark.sql.window import Window

fact_finance = spark.read.table("pl_workforce_analysis.gold_fact.fact_finance")
fact_payroll = spark.read.table("pl_workforce_analysis.gold_fact.fact_payroll")

dim_account = spark.read.table("pl_workforce_analysis.gold_dim.dim_account")
dim_employee = spark.read.table("pl_workforce_analysis.gold_dim.dim_employee")
dim_department = spark.read.table("pl_workforce_analysis.gold_dim.dim_department")

In [0]:
fact_finance = fact_finance.withColumn("amount_corrected",col("final_amount")*-1)
fact_finance = fact_finance.withColumn("month",col("fiscal_period").cast("int"))

In [0]:
# Monthly Consolidated Revenue

from pyspark.sql.functions import col, sum as _sum, round, lag, year
from pyspark.sql.window import Window

revenue_df = fact_finance .join(dim_account, "account_id") .filter(col("pnl_section") == "Revenue") .groupBy("fiscal_year", "month") .agg(round(_sum(col("amount_corrected")), 2).alias("revenue")) .orderBy("fiscal_year", "month")

window_spec = Window.orderBy("fiscal_year", "month")

revenue_df = revenue_df .withColumn("prev_revenue", lag("revenue").over(window_spec)) .withColumn("growth_pct",round(((col("revenue") - col("prev_revenue")) / col("prev_revenue")) * 100, 2 ))

display(revenue_df)

revenue_df.write.mode("overwrite").saveAsTable("pl_workforce_analysis.gold.monthly_revenue")


In [0]:
# Cost of Sales by Month 

from pyspark.sql.functions import col, sum as _sum, round

cogs_df = fact_finance .join(dim_account, "account_id").filter(col("pnl_section") == "COGS").groupBy("fiscal_year", "month") .agg(round(_sum(col("amount_corrected")), 2).alias("cogs")).orderBy("fiscal_year","month")

display(cogs_df)

cogs_df.write.mode("overwrite").saveAsTable("pl_workforce_analysis.gold.monthly_cogs")

In [0]:
# Monthly Gross Profit Margin

from pyspark.sql.functions import col, round

gross_df = revenue_df.join(cogs_df,["fiscal_year", "month"])

gross_df = gross_df.withColumn("gross_profit",col("revenue") - col("cogs")
).withColumn(
    "gross_margin_pct",
    round((col("gross_profit") / col("revenue")) * 100, 2)
)

display(gross_df.orderBy("fiscal_year", "month"))

gross_df.write.mode("overwrite").saveAsTable("pl_workforce_analysis.gold.monthly_gross_profit")

In [0]:
# Operating Expense Breakdown

from pyspark.sql.functions import col, sum as _sum, round, abs

opex_df = fact_finance.join(dim_account, "account_id") .filter(col("pnl_section") == "Operating Expense") .groupBy("company_id", "account_name") .agg(round(_sum(abs(col("final_amount"))), 2).alias("opex"))

display(opex_df)

opex_df.write.mode("overwrite").saveAsTable("pl_workforce_analysis.gold.opex_breakdown")

In [0]:
# Average Compensation by Position 

from pyspark.sql.functions import col, avg, round

avg_comp_df = fact_payroll.alias("fp") .join(dim_employee.alias("de"), "employee_id").groupBy(col("fp.company_id"), col("de.position")).agg(round(avg(col("fp.total_payroll_cost")), 2).alias("avg_compensation")) .orderBy(col("fp.company_id"), col("de.position"))

display(avg_comp_df)

avg_comp_df.write.mode("overwrite").saveAsTable("avg_compensation")

In [0]:
# Net Profit by Month 

from pyspark.sql.functions import col, sum as _sum, round, when

net_profit_df = fact_finance .join(dim_account, "account_id") .withColumn("adjusted_amount",when(col("pnl_section") == "Revenue", col("final_amount") * -1).otherwise(col("final_amount"))) .groupBy("fiscal_year", "month") .agg(round(_sum("adjusted_amount"), 2).alias("net_profit")) .orderBy("fiscal_year", "month")

display(net_profit_df)

net_profit_df.write.mode("overwrite").saveAsTable("pl_workforce_analysis.gold.monthly_net_profit")

In [0]:
# Overtime and Bonus Analysis

from pyspark.sql.functions import col
from pyspark.sql.functions import col, sum as _sum, round

fact_payroll = fact_payroll.withColumn("base_salary",col("gross_salary")- col("bonus")- col("overtime_pay")- col("commission")- col("allowances"))

overtime_bonus_df = fact_payroll.alias("fp").groupBy(col("fp.department_id")).agg(
    round(_sum(col("fp.overtime_pay")), 2).alias("total_overtime"),
    round(_sum(col("fp.bonus")), 2).alias("total_bonus"),
    round(
        _sum(col("fp.overtime_pay")) / _sum(col("fp.base_salary")),
        2
    ).alias("overtime_ratio")
) .orderBy(col("fp.department_id"))

display(overtime_bonus_df)

overtime_bonus_df.write.mode("overwrite").saveAsTable("overtime_bonus")

In [0]:
# Cost per Department

cost_dept_df = fact_payroll.alias("fp") .join(dim_department.alias("dd"), "department_id") .groupBy(col("fp.department_id")) .agg(round(_sum(col("fp.total_payroll_cost")), 2).alias("total_department_cost")) .orderBy(col("fp.department_id"))

display(cost_dept_df)

cost_dept_df.write.mode("overwrite").saveAsTable("pl_workforce_analysis.gold.cost_dept")

In [0]:
# Headcount Distribution by Department 

from pyspark.sql.functions import col, count

headcount_df = dim_employee.alias("de").filter(col("de.is_active") == True).groupBy(col("de.department_id")) .agg(count("employee_id").alias("headcount")) .orderBy(col("de.department_id"))

display(headcount_df)

headcount_df.write.mode("overwrite").saveAsTable("pl_workforce_analysis.gold.headcount_dept")

In [0]:
# Payroll Cost as % of Company Revenue

from pyspark.sql.functions import col, sum as _sum, round

payroll_df = fact_payroll.groupBy("company_id") .agg(_sum("total_payroll_cost").alias("total_payroll"))

revenue_df = fact_finance.alias("ff") .join(dim_account.alias("da"), "account_id") .filter(col("da.pnl_section") == "Revenue") .groupBy("ff.company_id") .agg(_sum("amount_corrected").alias("total_revenue"))

final_df = payroll_df.alias("p") .join(revenue_df.alias("r"), "company_id") .withColumn(
        "payroll_pct",
        round((col("p.total_payroll") / col("r.total_revenue")) * 100, 2)
    ) .orderBy("company_id")

display(final_df)

final_df.write.mode("overwrite").saveAsTable("pl_workforce_analysis.gold.payroll_revenue")